# Alpha Centauri: Structural Motif Discovery Across the AlphaFold Protein Universe

**ET-miner** | GPU-accelerated frequent itemset mining on 76.9M--109M proteins

---

This notebook presents a comprehensive analysis of proteome-scale pattern mining results:

- **Part 1** (Cells 1--9): Analysis of the 1K-feature "God Mode" campaign (26.8M itemsets, K=1--22)
- **Part 2** (Cells 10--15): 35K-feature Alpha Centauri results (16.8B itemsets, K=1--8)
- **Part 2.5**: K=8 Summary Statistics -- growth rates, support distributions across all K levels
- **Part 3** (Cells 16--18): Association rule mining and network analysis (1K + 35K)

**Dataset:** UniProt TrEMBL + SwissProt, 76.9M proteins (1K features) / 109.2M proteins (35K features)  
**Hardware:** NVIDIA H100/H200 GPUs, CSR bitvector encoding, Apriori with popcount-based support counting  
**Mining results:**
- 1K God Mode: 26.8M itemsets in 7.3 minutes (K=1--22)
- 35K Alpha Centauri: **16,812,646,639 itemsets** (K=1--8), including 12.07B K=8 itemsets (67 GB)

---

In [ ]:
# ============================================================================
# CELL 1: Setup + Data Loading
# ============================================================================

import json
import warnings
from pathlib import Path

import numpy as np
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore", category=FutureWarning)

# --- Color palette (consistent across all plots) ---
CATEGORY_COLORS = {
    "pfam": "#60a5fa",        # Blue
    "go_term": "#34d399",     # Green
    "plddt_mean": "#f472b6",  # Pink
    "plddt_fraction": "#fb923c",  # Orange
    # 35K categories
    "interpro": "#60a5fa",    # Blue (replaces pfam)
    "ec_number": "#a78bfa",   # Purple
    "keyword": "#fbbf24",     # Gold
    "taxonomy": "#f87171",    # Red
    "length_bin": "#94a3b8",  # Gray
    "plddt": "#f472b6",      # Pink
}

PLOTLY_TEMPLATE = "plotly_dark"

# --- Locate data files ---
BASE = Path(".").resolve()
REPO_ROOT = BASE
while not (REPO_ROOT / ".git").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

GODMODE_PATH = REPO_ROOT / "archived" / "alphafold" / "results_214m" / "itemsets_214m_godmode.parquet"
MAPPING_PATH = REPO_ROOT / "archived" / "alphafold" / "results_214m" / "item_mapping_214m.parquet"
SON_JSON = REPO_ROOT / "applications" / "alphafold" / "results_214m" / "experiment_direct_vs_son_20260219_050326.json"
NULL_JSON = REPO_ROOT / "applications" / "alphafold" / "results_214m" / "experiment_null_model_20260219_061046.json"
DECODED_TXT = REPO_ROOT / "applications" / "alphafold" / "results_214m" / "decoded_top_k_patterns.txt"

# --- Load core data ---
print("Loading God Mode itemsets (26.8M rows)...")
godmode = pl.read_parquet(str(GODMODE_PATH))
godmode = godmode.with_columns(pl.col("itemset").list.len().alias("k"))

print("Loading item mapping (1,006 features)...")
mapping = pl.read_parquet(str(MAPPING_PATH))

# Build lookup dict: item_id -> (feature_name, feature_category)
item_lookup = {
    row["item_id"]: (row["feature_name"], row["feature_category"])
    for row in mapping.iter_rows(named=True)
}

# --- Load experiment JSONs ---
with open(str(SON_JSON)) as f:
    son_data = json.load(f)
with open(str(NULL_JSON)) as f:
    null_data = json.load(f)

# --- Summary Statistics ---
N_PROTEINS = son_data["parameters"]["n_transactions"]  # 76,890,945

k_dist = (
    godmode.group_by("k")
    .agg(pl.len().alias("count"))
    .sort("k")
)

print("\n" + "=" * 72)
print("ALPHA CENTAURI -- GOD MODE MINING RESULTS")
print("=" * 72)
print(f"Proteins:          {N_PROTEINS:>14,}")
print(f"Features:          {mapping.height:>14,}")
print(f"Total itemsets:    {godmode.height:>14,}")
print(f"K range:           {1:>7} -- {godmode['k'].max()}")
print(f"Support range:     {godmode['support'].min():.2e} -- {godmode['support'].max():.6f}")
print(f"Mean support:      {godmode['support'].mean():.2e}")
print(f"Median support:    {godmode['support'].median():.2e}")
print("=" * 72)
print("\nK-Distribution:")
for row in k_dist.iter_rows():
    pct = row[1] / godmode.height * 100
    bar = "+" * max(1, int(pct))
    print(f"  K={row[0]:>2}: {row[1]:>10,} ({pct:5.1f}%)  {bar}")

# Feature category breakdown
cat_counts = mapping.group_by("feature_category").agg(pl.len().alias("count")).sort("count", descending=True)
print("\nFeature Categories:")
for row in cat_counts.iter_rows():
    print(f"  {row[0]:<20} {row[1]:>6}")

Loading God Mode itemsets (26.8M rows)...
Loading item mapping (1,006 features)...


In [ ]:
# ============================================================================
# CELL 2: K-Distribution Visualization
# ============================================================================
# Interactive Plotly bar chart: itemset count per K level (K=1 to K=22)
# Biological annotation per K range

k_data = k_dist.to_pandas()

# Biological theme annotations
bio_annotations = {
    2: "Basic pairwise associations (Pfam+pLDDT, GO pairs)",
    5: "Multi-domain architectures emerge",
    9: "Peak: complex functional modules",
    14: "Deep motifs: multi-pathway hubs",
    19: "RNA helicase / spliceosome sentinel",
    22: "Deepest: 22-feature signature (1 itemset, ~8 proteins)",
}

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("K-Distribution (Linear Scale)", "K-Distribution (Log Scale)"),
    horizontal_spacing=0.08,
)

# Color gradient by K
colors = [f"hsl({int(200 + k * 6)}, 70%, {max(30, 75 - k * 2)}%)" for k in k_data["k"]]

for col_idx, log_y in enumerate([False, True], 1):
    fig.add_trace(
        go.Bar(
            x=k_data["k"],
            y=k_data["count"],
            marker_color=colors,
            text=[f"{c:,}" for c in k_data["count"]],
            textposition="outside",
            textfont=dict(size=8),
            hovertemplate="K=%{x}<br>Count: %{y:,}<br>%{customdata}<extra></extra>",
            customdata=[
                bio_annotations.get(k, f"{c / godmode.height * 100:.1f}% of total")
                for k, c in zip(k_data["k"], k_data["count"])
            ],
            showlegend=False,
        ),
        row=1, col=col_idx,
    )
    if log_y:
        fig.update_yaxes(type="log", row=1, col=col_idx)

# Add bio-theme annotations on log plot
for k_val, text in bio_annotations.items():
    count_at_k = k_data.loc[k_data["k"] == k_val, "count"]
    if len(count_at_k) > 0:
        fig.add_annotation(
            x=k_val, y=np.log10(float(count_at_k.iloc[0])),
            text=text.split(":")[0] if ":" in text else text[:35] + "...",
            showarrow=True, arrowhead=2, arrowsize=0.8,
            ax=40, ay=-30 if k_val < 15 else 30,
            font=dict(size=9, color="#94a3b8"),
            arrowcolor="#475569",
            row=1, col=2,
        )

fig.update_layout(
    title=dict(
        text="God Mode Mining: 26.8M Itemsets Across K=1--22<br>"
             "<sub>76.9M proteins, 1,002 features (500 Pfam + 500 GO + 6 pLDDT), support >= 1e-7</sub>",
        font=dict(size=18),
    ),
    template=PLOTLY_TEMPLATE,
    height=500,
    width=1200,
    xaxis_title="K (itemset size)",
    yaxis_title="Number of itemsets",
    xaxis2_title="K (itemset size)",
    yaxis2_title="Number of itemsets (log)",
)

fig.show()

### Biological Interpretation: K-Distribution

The K-distribution reveals a clear biological hierarchy:

| K Range | Itemsets | Biological Theme |
|---------|---------|------------------|
| K=1--3 | 527K | Individual features and basic pairings: Pfam+pLDDT quality, simple GO co-occurrences |
| K=4--6 | 5.8M | Multi-domain architectures: kinase+ATP-binding+membrane, transporter+channel+signal peptide |
| K=7--9 | **10.1M (peak)** | Complex functional modules: entire pathway signatures, multi-enzyme complexes |
| K=10--14 | 6.0M | Deep functional coupling: cross-pathway hubs, ribosomal machinery, spliceosome components |
| K=15--22 | 5.5K | Ultra-rare sentinels: the deepest co-occurrence patterns, found in ~100--200 proteins each |

The **peak at K=9** (3.53M itemsets) means the proteome's most common "functional complexity" involves 9 co-occurring features. This is not random: the null model produces **zero** itemsets at K >= 7.

In [ ]:
# ============================================================================
# CELL 3: Support Distribution (Rank vs Support -- Power Law / Zipf Analysis)
# ============================================================================

# Sort all itemsets by support (descending) for rank plot
# Use a subsample for plotting performance (26.8M points is too many)
support_sorted = godmode.select("support").sort("support", descending=True)

# Log-spaced subsample: keep top 10K, then log-space the rest
n_total = support_sorted.height
top_n = min(10_000, n_total)
log_indices = np.unique(np.logspace(np.log10(top_n), np.log10(n_total - 1), num=50_000).astype(int))
sample_indices = np.concatenate([np.arange(top_n), log_indices])
sample_indices = np.unique(sample_indices)

ranks = sample_indices + 1  # 1-indexed rank
supports = support_sorted["support"].gather(sample_indices.tolist()).to_numpy()

# Fit power law: log(support) = alpha * log(rank) + beta
log_r = np.log10(ranks.astype(float))
log_s = np.log10(supports + 1e-15)  # Avoid log(0)

# Fit on middle 80% to avoid edge effects
fit_mask = (log_r > np.percentile(log_r, 10)) & (log_r < np.percentile(log_r, 90))
coeffs = np.polyfit(log_r[fit_mask], log_s[fit_mask], 1)
alpha, beta = coeffs
fit_line = 10 ** (alpha * log_r + beta)

# R-squared
ss_res = np.sum((log_s[fit_mask] - (alpha * log_r[fit_mask] + beta)) ** 2)
ss_tot = np.sum((log_s[fit_mask] - np.mean(log_s[fit_mask])) ** 2)
r_squared = 1 - ss_res / ss_tot

fig = go.Figure()

fig.add_trace(go.Scattergl(
    x=ranks,
    y=supports,
    mode="markers",
    marker=dict(size=2, color="#60a5fa", opacity=0.3),
    name="Itemsets",
    hovertemplate="Rank: %{x:,}<br>Support: %{y:.2e}<extra></extra>",
))

fig.add_trace(go.Scatter(
    x=ranks,
    y=fit_line,
    mode="lines",
    line=dict(color="#f87171", width=2, dash="dash"),
    name=f"Power law fit (alpha={alpha:.3f}, R2={r_squared:.4f})",
))

fig.update_layout(
    title=dict(
        text=f"Zipf Analysis: Rank vs Support (26.8M itemsets)<br>"
             f"<sub>Power-law exponent alpha = {alpha:.3f}, R-squared = {r_squared:.4f}</sub>",
        font=dict(size=16),
    ),
    xaxis=dict(title="Rank", type="log"),
    yaxis=dict(title="Support (fraction of 76.9M proteins)", type="log"),
    template=PLOTLY_TEMPLATE,
    height=550,
    width=1000,
    legend=dict(x=0.02, y=0.02, xanchor="left", yanchor="bottom"),
)

# Annotate notable points
fig.add_annotation(
    x=0, y=np.log10(supports[0]),
    text=f"Rank 1: support={supports[0]:.4f} (59.6% of proteins)",
    showarrow=True, arrowhead=2, ax=120, ay=-30,
    font=dict(size=10, color="#fbbf24"),
)

fig.show()

print(f"\nPower-law fit: support ~ rank^{alpha:.3f}")
print(f"R-squared: {r_squared:.4f}")
print(f"Zipf exponent: {-alpha:.3f} (classic Zipf = 1.0)")
print(f"Support spans {supports[0] / supports[-1]:.0e}x dynamic range")

### Interpretation: Zipf / Power-Law Structure

The rank-support plot reveals a strong power-law relationship, consistent with Zipf's law observed in many biological systems (gene expression, protein abundance, metabolite concentrations). The exponent being steeper than classic Zipf (|alpha| > 1) reflects the exponential combinatorial explosion at higher K: as patterns grow deeper, they become exponentially rarer.

This is not merely a statistical artifact of the Apriori algorithm -- the null model (random permutations) produces a **fundamentally different** distribution that dies at K=6. The real data's long tail extending to K=22 is a signature of genuine biological organization.

In [ ]:
# ============================================================================
# CELL 4: Feature Frequency Analysis
# ============================================================================
# Which features appear in the most itemsets? Top features by frequency,
# colored by feature category. Feature "depth" analysis: average K-level.

# Explode itemsets to count feature occurrences
# For 26.8M itemsets this is heavy -- use streaming approach
print("Computing feature frequencies across 26.8M itemsets...")

# Count how often each item_id appears across all itemsets
feature_counts = (
    godmode
    .select("itemset", "k")
    .explode("itemset")
    .group_by("itemset")
    .agg(
        pl.len().alias("frequency"),
        pl.col("k").mean().alias("avg_k"),
        pl.col("k").max().alias("max_k"),
        pl.col("k").min().alias("min_k"),
    )
    .sort("frequency", descending=True)
    .rename({"itemset": "item_id"})
)

# Join with mapping for names and categories
feature_stats = feature_counts.join(mapping, on="item_id", how="left")

print(f"\nAll {feature_stats.height} features analyzed.")
print("\nTop 30 features by itemset frequency:")
print(feature_stats.head(30).select(
    "feature_name", "feature_category", "frequency", "avg_k", "max_k"
))

# --- Plot: Top 40 features by frequency, colored by category ---
top_n = 40
top_feats = feature_stats.head(top_n).to_pandas()
top_feats["color"] = top_feats["feature_category"].map(CATEGORY_COLORS).fillna("#94a3b8")

fig = go.Figure()

for cat in top_feats["feature_category"].unique():
    mask = top_feats["feature_category"] == cat
    subset = top_feats[mask]
    fig.add_trace(go.Bar(
        x=subset["feature_name"],
        y=subset["frequency"],
        name=cat,
        marker_color=CATEGORY_COLORS.get(cat, "#94a3b8"),
        hovertemplate=(
            "%{x}<br>"
            f"Category: {cat}<br>"
            "Frequency: %{y:,}<br>"
            "Avg K: %{customdata[0]:.1f}<br>"
            "Max K: %{customdata[1]}<extra></extra>"
        ),
        customdata=subset[["avg_k", "max_k"]].values,
    ))

fig.update_layout(
    title=dict(
        text=f"Top {top_n} Features by Itemset Frequency<br>"
             "<sub>How often each feature participates in a mined pattern</sub>",
        font=dict(size=16),
    ),
    xaxis=dict(title="Feature", tickangle=45, tickfont=dict(size=8)),
    yaxis=dict(title="Itemset frequency (out of 26.8M)"),
    template=PLOTLY_TEMPLATE,
    height=600,
    width=1200,
    barmode="stack",
    legend=dict(title="Category"),
)
fig.show()

# --- Feature Depth Analysis: avg K per feature, by category ---
fig2 = px.scatter(
    feature_stats.to_pandas(),
    x="avg_k",
    y="frequency",
    color="feature_category",
    color_discrete_map=CATEGORY_COLORS,
    hover_name="feature_name",
    hover_data={"max_k": True, "frequency": ":,"},
    log_y=True,
    title="Feature Depth vs Frequency<br>"
          "<sub>Average K-level each feature participates in vs total itemset count</sub>",
    labels={"avg_k": "Average K-level", "frequency": "Itemset frequency (log)"},
    template=PLOTLY_TEMPLATE,
    height=500,
    width=1000,
)
fig2.show()

In [ ]:
# ============================================================================
# CELL 5: Feature Co-occurrence Heatmap
# ============================================================================
# Cluster features by co-occurrence patterns across itemsets.
# We use the top 60 most frequent features (full 1002 x 1002 is too large).

from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform

TOP_N_FEATURES = 60  # Top features for heatmap

top_ids = feature_stats.head(TOP_N_FEATURES)["item_id"].to_list()
top_names = [
    item_lookup.get(i, (f"item_{i}", "unknown"))[0]
    for i in top_ids
]
top_categories = [
    item_lookup.get(i, (f"item_{i}", "unknown"))[1]
    for i in top_ids
]

# Build co-occurrence matrix from itemsets that contain at least 2 top features
print(f"Computing {TOP_N_FEATURES}x{TOP_N_FEATURES} co-occurrence matrix...")
id_set = set(top_ids)
id_to_idx = {fid: i for i, fid in enumerate(top_ids)}
cooc = np.zeros((TOP_N_FEATURES, TOP_N_FEATURES), dtype=np.int64)

# Process in chunks to avoid memory explosion on 26.8M rows
CHUNK_SIZE = 2_000_000
for offset in range(0, godmode.height, CHUNK_SIZE):
    chunk = godmode.slice(offset, min(CHUNK_SIZE, godmode.height - offset))
    for itemset in chunk["itemset"].to_list():
        present = [id_to_idx[x] for x in itemset if x in id_set]
        for i in range(len(present)):
            for j in range(i + 1, len(present)):
                cooc[present[i], present[j]] += 1
                cooc[present[j], present[i]] += 1

# Diagonal = self-frequency
for idx, fid in enumerate(top_ids):
    freq = feature_stats.filter(pl.col("item_id") == fid)["frequency"][0]
    cooc[idx, idx] = freq

# Normalize to Jaccard-like similarity: co-occurrence / (freq_i + freq_j - co-occurrence)
diag = np.diag(cooc).astype(float)
jaccard = np.zeros_like(cooc, dtype=float)
for i in range(TOP_N_FEATURES):
    for j in range(TOP_N_FEATURES):
        denom = diag[i] + diag[j] - cooc[i, j]
        jaccard[i, j] = cooc[i, j] / denom if denom > 0 else 0.0

# Hierarchical clustering for ordering
dist_matrix = 1.0 - jaccard
np.fill_diagonal(dist_matrix, 0)
dist_matrix = np.clip(dist_matrix, 0, None)
dist_condensed = squareform(dist_matrix, checks=False)
linkage_matrix = linkage(dist_condensed, method="ward")
order = leaves_list(linkage_matrix)

# Reorder
jaccard_ordered = jaccard[order][:, order]
names_ordered = [top_names[i] for i in order]
cats_ordered = [top_categories[i] for i in order]

# Category color bar
cat_color_bar = [CATEGORY_COLORS.get(c, "#94a3b8") for c in cats_ordered]

fig = go.Figure(data=go.Heatmap(
    z=jaccard_ordered,
    x=names_ordered,
    y=names_ordered,
    colorscale="Viridis",
    colorbar=dict(title="Jaccard<br>Similarity"),
    hovertemplate="%{x} -- %{y}<br>Jaccard: %{z:.3f}<extra></extra>",
))

fig.update_layout(
    title=dict(
        text=f"Feature Co-occurrence Heatmap (Top {TOP_N_FEATURES}, Ward Clustering)<br>"
             "<sub>Jaccard similarity of co-occurrence across 26.8M itemsets</sub>",
        font=dict(size=16),
    ),
    template=PLOTLY_TEMPLATE,
    height=900,
    width=1000,
    xaxis=dict(tickangle=45, tickfont=dict(size=7)),
    yaxis=dict(tickfont=dict(size=7)),
)

fig.show()

print("\nCluster ordering applied (Ward linkage on 1-Jaccard distance).")
print("Features that co-occur frequently are grouped together.")

In [ ]:
# ============================================================================
# CELL 6: Itemset Diversity per K-level (Jaccard Similarity)
# ============================================================================
# At each K, compute pairwise Jaccard similarity between itemsets.
# High similarity = redundant patterns. Low = diverse discoveries.

from itertools import combinations as combs

MAX_SAMPLE_PER_K = 1000  # Sample for tractable pairwise computation
MAX_PAIRS = 50_000

k_diversity = []

for k_val in range(2, 23):
    k_items = godmode.filter(pl.col("k") == k_val)
    n_k = k_items.height
    if n_k < 2:
        continue

    # Sample if needed
    if n_k > MAX_SAMPLE_PER_K:
        k_items = k_items.sample(MAX_SAMPLE_PER_K, seed=42)

    # Convert to sets
    itemsets_as_sets = [set(row) for row in k_items["itemset"].to_list()]

    # Compute pairwise Jaccard
    jaccards = []
    pairs = list(combs(range(len(itemsets_as_sets)), 2))
    if len(pairs) > MAX_PAIRS:
        rng = np.random.RandomState(42)
        idx = rng.choice(len(pairs), MAX_PAIRS, replace=False)
        pairs = [pairs[i] for i in idx]

    for i, j in pairs:
        inter = len(itemsets_as_sets[i] & itemsets_as_sets[j])
        union = len(itemsets_as_sets[i] | itemsets_as_sets[j])
        jaccards.append(inter / union if union > 0 else 0.0)

    jaccards = np.array(jaccards)
    k_diversity.append({
        "k": k_val,
        "n_itemsets": n_k,
        "mean_jaccard": float(np.mean(jaccards)),
        "std_jaccard": float(np.std(jaccards)),
        "median_jaccard": float(np.median(jaccards)),
    })
    print(f"K={k_val:2d}: n={n_k:>10,}, mean_J={np.mean(jaccards):.4f} +/- {np.std(jaccards):.4f}")

div_df = pl.DataFrame(k_diversity).to_pandas()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=div_df["k"],
    y=div_df["mean_jaccard"],
    mode="lines+markers",
    name="Mean Jaccard",
    line=dict(color="#60a5fa", width=3),
    marker=dict(size=8),
    error_y=dict(
        type="data",
        array=div_df["std_jaccard"],
        visible=True,
        color="rgba(96,165,250,0.3)",
    ),
))

fig.add_trace(go.Scatter(
    x=div_df["k"],
    y=div_df["median_jaccard"],
    mode="lines+markers",
    name="Median Jaccard",
    line=dict(color="#34d399", width=2, dash="dot"),
    marker=dict(size=6),
))

# Interpretation zones
fig.add_hrect(y0=0.7, y1=1.0, fillcolor="rgba(248,113,113,0.1)",
              line_width=0, annotation_text="High redundancy", annotation_position="top left")
fig.add_hrect(y0=0.0, y1=0.3, fillcolor="rgba(52,211,153,0.1)",
              line_width=0, annotation_text="High diversity", annotation_position="bottom left")

fig.update_layout(
    title=dict(
        text="Itemset Diversity per K-Level<br>"
             "<sub>Pairwise Jaccard similarity between itemsets at each K (sampled, max 1K itemsets)</sub>",
        font=dict(size=16),
    ),
    xaxis=dict(title="K (itemset size)", dtick=1),
    yaxis=dict(title="Jaccard Similarity", range=[0, 1]),
    template=PLOTLY_TEMPLATE,
    height=500,
    width=1000,
)
fig.show()

### Interpretation: Diversity vs Redundancy

**Low Jaccard at low K:** Small itemsets are diverse -- many different pairs/triples of features co-occur independently.

**Rising Jaccard at high K:** As K increases, itemsets converge toward a common "core" of features. The K=19 itemsets are all minor variations of a single RNA helicase / spliceosome signature (sharing 16--17 of 19 features). This convergence is biologically meaningful: at high K, the patterns represent tightly coupled functional modules where every feature is nearly obligate.

The transition from diverse (K < 7) to convergent (K > 12) marks the boundary between combinatorial feature exploration and true functional constraint.

In [ ]:
# ============================================================================
# CELL 7: UMAP Embedding of High-K Itemsets
# ============================================================================
# Take itemsets with K >= 8, create binary feature vectors,
# UMAP embedding, color by K level.

try:
    from umap import UMAP
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print("UMAP not installed. Install with: uv pip install umap-learn")
    print("Falling back to t-SNE from sklearn.")

from sklearn.manifold import TSNE

K_MIN_EMBED = 8
MAX_EMBED = 15_000  # Max points for embedding

high_k = godmode.filter(pl.col("k") >= K_MIN_EMBED)
print(f"High-K itemsets (K >= {K_MIN_EMBED}): {high_k.height:,}")

if high_k.height > MAX_EMBED:
    high_k = high_k.sample(MAX_EMBED, seed=42)
    print(f"Sampled to {MAX_EMBED:,} for embedding")

# Build binary feature matrix
n_features = mapping.height
item_ids_list = high_k["itemset"].to_list()
k_vals = high_k["k"].to_list()
supports = high_k["support"].to_list()

X = np.zeros((len(item_ids_list), n_features), dtype=np.uint8)
for i, itemset in enumerate(item_ids_list):
    for item_id in itemset:
        if 0 <= item_id < n_features:
            X[i, item_id] = 1

print(f"Feature matrix shape: {X.shape}")
print(f"Sparsity: {1 - X.sum() / X.size:.1%}")

# Embed
print("Running embedding (this may take 1-3 minutes)...")
if HAS_UMAP:
    reducer = UMAP(n_components=2, n_neighbors=30, min_dist=0.1, metric="jaccard", random_state=42)
    embedding = reducer.fit_transform(X)
    method_name = "UMAP"
else:
    reducer = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
    embedding = reducer.fit_transform(X.astype(float))
    method_name = "t-SNE"

print(f"{method_name} embedding complete.")

# Build hover text with decoded features
hover_texts = []
for itemset in item_ids_list:
    decoded = [item_lookup.get(x, (f"item_{x}", "?"))[0] for x in itemset[:8]]  # Limit hover size
    suffix = f" +{len(itemset)-8} more" if len(itemset) > 8 else ""
    hover_texts.append(", ".join(decoded) + suffix)

fig = go.Figure()

# Color by K
for k_val in sorted(set(k_vals)):
    mask = [k == k_val for k in k_vals]
    idx = [i for i, m in enumerate(mask) if m]
    fig.add_trace(go.Scattergl(
        x=embedding[idx, 0],
        y=embedding[idx, 1],
        mode="markers",
        marker=dict(
            size=4 + (k_val - K_MIN_EMBED),
            opacity=0.6,
        ),
        name=f"K={k_val} (n={len(idx):,})",
        text=[hover_texts[i] for i in idx],
        hovertemplate="K=%{meta}<br>%{text}<extra></extra>",
        meta=[k_val] * len(idx),
    ))

fig.update_layout(
    title=dict(
        text=f"{method_name} Embedding of High-K Itemsets (K >= {K_MIN_EMBED})<br>"
             f"<sub>{len(item_ids_list):,} itemsets embedded by binary feature vectors, colored by K</sub>",
        font=dict(size=16),
    ),
    xaxis=dict(title=f"{method_name}-1", showgrid=False),
    yaxis=dict(title=f"{method_name}-2", showgrid=False),
    template=PLOTLY_TEMPLATE,
    height=700,
    width=1000,
    legend=dict(title="K-level"),
)
fig.show()

In [ ]:
# ============================================================================
# CELL 8: SON vs Direct GPU Comparison
# ============================================================================
# What does the approximate SON algorithm miss compared to exact Direct GPU?

direct_k = son_data["direct_gpu_result"]["k_distribution"]
direct_total = son_data["direct_gpu_result"]["itemsets"]
direct_time = son_data["direct_gpu_result"]["time_seconds"]

son_total = son_data["son_reference"]["itemsets"]
son_time = son_data["son_reference"]["time_seconds"]
son_max_k = son_data["son_reference"]["max_k"]

# SON does not have per-K distribution in the JSON -- compute what we can
# The comparison shows SON lost 95.2% of itemsets (453,019 out of 475,865)
speedup = son_data["comparison"]["speedup_vs_son"]
lost = son_data["comparison"]["itemset_diff"]
pct_lost = lost / direct_total * 100

# Build per-K comparison data
k_vals = sorted(direct_k.keys(), key=int)
direct_counts = [direct_k[k] for k in k_vals]
k_ints = [int(k) for k in k_vals]

# SON approximation: it found 22,846 total vs 475,865 direct
# SON max K = 13, Direct max K = 14
# We know SON primarily loses high-K patterns due to chunk-level threshold inflation

print("=" * 60)
print("DIRECT GPU vs SON COMPARISON")
print("=" * 60)
print(f"Support threshold:   {son_data['parameters']['support_pct']}")
print(f"Proteins:            {son_data['parameters']['n_transactions']:,}")
print(f"")
print(f"Direct GPU:          {direct_total:>10,} itemsets in {direct_time:.1f}s (max K={son_data['direct_gpu_result']['max_k']})")
print(f"SON (approximate):   {son_total:>10,} itemsets in {son_time:.1f}s (max K={son_max_k})")
print(f"")
print(f"Speedup:             {speedup:.1f}x")
print(f"SON lost:            {lost:,} itemsets ({pct_lost:.1f}%)")

# --- Waterfall chart: what SON misses ---
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f"Direct GPU vs SON: Itemsets per K (0.001% support)",
        f"SON Coverage Waterfall"
    ),
    horizontal_spacing=0.12,
)

# Left panel: Direct GPU K-distribution
fig.add_trace(go.Bar(
    x=k_ints,
    y=direct_counts,
    name="Direct GPU",
    marker_color="#60a5fa",
    text=[f"{c:,}" for c in direct_counts],
    textposition="outside",
    textfont=dict(size=8),
), row=1, col=1)

fig.update_yaxes(type="log", row=1, col=1)

# Right panel: Waterfall showing what's lost
# Since we only have totals (not SON per-K), show the aggregate comparison
categories = ["Direct GPU", "SON found", "Lost to\nchunk inflation"]
values = [direct_total, son_total, -lost]
colors = ["#60a5fa", "#34d399", "#f87171"]

fig.add_trace(go.Waterfall(
    x=categories,
    y=[direct_total, -(direct_total - son_total), 0],
    measure=["absolute", "relative", "total"],
    text=[f"{direct_total:,}", f"-{lost:,}", f"{son_total:,}"],
    textposition="outside",
    connector=dict(line=dict(color="#475569")),
    increasing=dict(marker_color="#60a5fa"),
    decreasing=dict(marker_color="#f87171"),
    totals=dict(marker_color="#34d399"),
), row=1, col=2)

fig.update_layout(
    title=dict(
        text=f"Direct GPU vs SON: {speedup:.0f}x Faster, {pct_lost:.1f}% More Complete<br>"
             f"<sub>SON lost {lost:,} of {direct_total:,} itemsets due to chunk-level support threshold inflation</sub>",
        font=dict(size=16),
    ),
    template=PLOTLY_TEMPLATE,
    height=500,
    width=1200,
    showlegend=False,
)

fig.show()

# Also show blitz comparison
blitz = son_data["blitz_reference"]
print(f"\nBlitz mode (0.0001% support):")
print(f"  {blitz['itemsets']:,} itemsets in {blitz['time_seconds']:.1f}s (max K={blitz['max_k']})")

### Why Direct GPU Beats SON

The SON (Savasere-Omiecinski-Navathe) algorithm splits data into chunks and mines each independently, then validates candidates globally. The problem: chunk-level support thresholds are **inflated** to ensure no false negatives, but this creates massive false negative losses in practice.

At 0.001% support:
- **Direct GPU finds 475,865 itemsets** in 50.7 seconds
- **SON finds only 22,846** in 1,085.6 seconds (21.4x slower!)
- SON **loses 95.2%** of all patterns, primarily the high-K discoveries

The ET-miner CSR bitvector approach makes SON unnecessary: the entire transaction database fits in GPU memory as a compressed bitvector, enabling exact global counting with no approximation.

In [ ]:
# ============================================================================
# CELL 9: Null Model Overlay
# ============================================================================
# Real vs permuted (null) K-distributions. Z-score annotations per K.
# Visual proof that K >= 7 is biologically significant (null = zero).

real_dist = null_data["real_distribution"]
stats = null_data["statistics"]
null_runs = null_data["null_runs"]

# Collect null distributions across runs
all_k = sorted(real_dist.keys(), key=int)
k_ints = [int(k) for k in all_k]
real_counts = [real_dist[k] for k in all_k]

null_means = [stats[k]["null_mean"] for k in all_k]
null_stds = [stats[k]["null_std"] for k in all_k]
z_scores = [stats[k]["z_score"] for k in all_k]
directions = [stats[k]["direction"] for k in all_k]

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=(
        "K-Distribution: Real Data vs Null Model (5 permutations)",
        "Z-Scores per K-Level (significance)",
    ),
    vertical_spacing=0.15,
    row_heights=[0.6, 0.4],
)

# --- Top: overlapping K-distributions ---
# Real data
fig.add_trace(go.Bar(
    x=k_ints,
    y=real_counts,
    name="Real (biological)",
    marker_color="#60a5fa",
    opacity=0.8,
    text=[f"{c:,}" for c in real_counts],
    textposition="outside",
    textfont=dict(size=7),
), row=1, col=1)

# Null model mean + error bars
fig.add_trace(go.Bar(
    x=k_ints,
    y=null_means,
    name="Null (permuted)",
    marker_color="#f87171",
    opacity=0.7,
    error_y=dict(
        type="data",
        array=null_stds,
        visible=True,
        color="rgba(248,113,113,0.5)",
    ),
), row=1, col=1)

# Individual null runs as ghost lines
for run in null_runs:
    run_k = sorted(run["k_distribution"].keys(), key=int)
    run_counts = [run["k_distribution"][k] for k in run_k]
    fig.add_trace(go.Scatter(
        x=[int(k) for k in run_k],
        y=run_counts,
        mode="lines",
        line=dict(color="rgba(248,113,113,0.15)", width=1),
        showlegend=False,
        hoverinfo="skip",
    ), row=1, col=1)

fig.update_yaxes(type="log", row=1, col=1)

# --- Bottom: Z-scores ---
# Cap infinite Z-scores for display
z_display = []
z_colors = []
z_text = []
for k, z, d in zip(k_ints, z_scores, directions):
    if z == "inf":
        z_display.append(100_000)  # Capped for visualization
        z_text.append("Z = INF")
        z_colors.append("#34d399")
    elif isinstance(z, (int, float)):
        z_val = float(z)
        z_display.append(z_val)
        z_text.append(f"Z = {z_val:,.0f}")
        if z_val > 0:
            z_colors.append("#34d399" if z_val > 100 else "#fbbf24")
        else:
            z_colors.append("#f87171")
    else:
        z_display.append(0)
        z_text.append(f"Z = {z}")
        z_colors.append("#94a3b8")

fig.add_trace(go.Bar(
    x=k_ints,
    y=z_display,
    marker_color=z_colors,
    text=z_text,
    textposition="outside",
    textfont=dict(size=8),
    showlegend=False,
    hovertemplate="K=%{x}<br>%{text}<extra></extra>",
), row=2, col=1)

fig.update_yaxes(type="log", row=2, col=1)

# Significance line at Z=3.29 (p < 0.001)
fig.add_hline(y=3.29, line_dash="dash", line_color="#fbbf24",
              annotation_text="Z=3.29 (p<0.001)", row=2, col=1)

fig.update_layout(
    title=dict(
        text="Null Model Permutation Test: Real Biology vs Random Chance<br>"
             f"<sub>{null_data['parameters']['n_permutations']} permutations, "
             f"support={null_data['parameters']['min_support']:.0e}, "
             f"{null_data['summary']['total_experiment_seconds']:.0f}s total</sub>",
        font=dict(size=16),
    ),
    template=PLOTLY_TEMPLATE,
    height=800,
    width=1100,
    barmode="group",
    xaxis=dict(title="K"),
    xaxis2=dict(title="K"),
    yaxis=dict(title="Itemset count (log)"),
    yaxis2=dict(title="Z-score (log)"),
)

fig.show()

# Summary table
print("\n" + "=" * 72)
print("NULL MODEL SIGNIFICANCE SUMMARY")
print("=" * 72)
print(f"{'K':>3}  {'Real':>10}  {'Null mean':>10}  {'Z-score':>12}  {'Direction':>10}")
print("-" * 55)
for k in all_k:
    s = stats[k]
    z = s['z_score']
    z_str = f"{z:>12,.1f}" if isinstance(z, (int, float)) else f"{'INF':>12}"
    print(f"{int(k):>3}  {s['real']:>10,}  {s['null_mean']:>10,.1f}  {z_str}  {s['direction']:>10}")
print(f"\nKey result: K >= 7 has Z = INFINITY (null produces exactly 0 itemsets)")
print(f"All {sum(real_dist[k] for k in all_k if int(k) >= 7):,} itemsets at K >= 7 are PURELY biological.")

### Interpretation: The Null Model Boundary

The null model (random permutation of feature assignments) establishes a critical boundary:

- **K=1:** Identical by construction (1,002/1,002 features, sanity check)
- **K=2--3:** Null **exceeds** real data (Z = -987, -143). Random co-occurrence from skewed feature frequencies inflates shallow pairs. The real proteome has **fewer** K=2--3 patterns than chance because biological constraints **prevent** certain combinations.
- **K=4:** Crossover point. Real data starts exceeding null (Z = 3,791).
- **K=5:** Strong enrichment (Z = 7,402). Multi-domain architectures that evolution built.
- **K=6:** Extreme enrichment (Z = 71,728). Null barely reaches K=6 (21.8 itemsets vs 78,596 real).
- **K >= 7:** The null model produces **exactly zero** itemsets. Z = infinity. Every single one of the 89,566 real itemsets at K >= 7 represents genuine biological organization that random chance cannot produce.

This is the strongest possible statistical validation: not merely significant, but **impossible** under the null hypothesis.

---

# Part 2: 35K-Feature Alpha Centauri Results

**Mining complete.** 16,812,646,639 frequent itemsets mined across K=1--8 from 109.2M proteins with 35,012 features.

| K | Itemsets | Size |
|---|---------|------|
| 1 | 28,405 | tiny |
| 2 | 5,506,372 | tiny |
| 3 | 176,048,236 | ~3 GB |
| 4 | 1,506,508,703 | ~8 GB |
| 5 | 2,474,423,427 | ~10 GB |
| 6 | 626,781,137 | ~4 GB |
| 7 | 3,507,040,364 | 16.6 GB |
| 8 | 12,072,309,005 | 67 GB |
| **Total** | **16,812,646,639** | |

**Note:** K=7 and K=8 require streaming/row-group iteration due to size. K=8 exceeds Polars u32 row limit.

---

In [ ]:
# ============================================================================
# CELL 10: 35K Data Loading — Environment Detection + Multi-K Parquets
# ============================================================================
# K=1-6: eager load (small enough for memory)
# K=7 (16.6 GB, 3.5B rows): Polars streaming or PyArrow row-group iteration
# K=8 (67 GB, 12B rows): PyArrow row-group iteration ONLY (exceeds Polars u32 row limit)

import pyarrow.parquet as pq

# --- Environment auto-detection ---
# Local (WSL2): data on /mnt/d/
# Remote (Vast.ai): data in /workspace/
if Path("/workspace").exists():
    _BASE = Path("/workspace/et-miner-data")
    ENV_NAME = "remote (Vast.ai)"
elif Path("/mnt/d/et-miner-data").exists():
    _BASE = Path("/mnt/d/et-miner-data")
    ENV_NAME = "local (WSL2)"
else:
    # Fallback: relative path from notebook location
    _BASE = Path("../../mnt/d/et-miner-data")
    ENV_NAME = "local (relative)"

RESULTS_DIR = _BASE / "results_35k"
K1_K6_DIR = RESULTS_DIR / "run3" / "parquet"  # K=1-6 identical in run3 and run4
K7_PATH = RESULTS_DIR / "run3" / "parquet" / "frequent_k7.parquet"  # 16.6 GB
K8_PATH = RESULTS_DIR / "run4" / "parquet" / "frequent_k8.parquet"  # 67 GB
ITEM_MAPPING_35K = _BASE / "alpha-centauri-35k" / "item_mapping_35k.parquet"

print(f"Environment: {ENV_NAME}")
print(f"Results dir: {RESULTS_DIR}")
print(f"  exists: {RESULTS_DIR.exists()}")

# --- Load item mapping ---
HAS_35K_RESULTS = RESULTS_DIR.exists()
HAS_K8_DATA = K8_PATH.exists()

if ITEM_MAPPING_35K.exists():
    mapping_35k = pl.read_parquet(str(ITEM_MAPPING_35K))
    print(f"\n35K mapping loaded: {mapping_35k.shape}")

    cats = mapping_35k.group_by("feature_category").agg(pl.len().alias("count")).sort("count", descending=True)
    print(f"\nFeature categories:")
    for row in cats.iter_rows():
        print(f"  {row[0]:<20} {row[1]:>6,}")

    # Build lookup dicts (reusing nominate_targets.py pattern)
    item_lookup_35k = {
        row["item_id"]: (row["feature_name"], row["feature_category"])
        for row in mapping_35k.iter_rows(named=True)
    }
    id_to_group_35k = {
        row["item_id"]: row["feature_category"]
        for row in mapping_35k.iter_rows(named=True)
    }
else:
    print(f"WARNING: Item mapping not found at {ITEM_MAPPING_35K}")
    item_lookup_35k = {}
    id_to_group_35k = {}

# --- Load K=1-6 parquets (eager, fits in memory) ---
data_by_k: dict[int, pl.DataFrame] = {}

if HAS_35K_RESULTS:
    for k in range(1, 7):
        path = K1_K6_DIR / f"frequent_k{k}.parquet"
        if path.exists():
            df = pl.read_parquet(str(path))
            # Ensure 'k' column exists
            if "k" not in df.columns:
                df = df.with_columns(pl.lit(k).alias("k").cast(pl.UInt8))
            data_by_k[k] = df
            print(f"  K={k}: {df.height:>14,} rows loaded (eager)")
        else:
            print(f"  K={k}: NOT FOUND at {path}")

    # K=7: Polars streaming (stays lazy until needed)
    if K7_PATH.exists():
        k7_lazy = pl.scan_parquet(str(K7_PATH))
        print(f"  K=7: lazy scan ready ({K7_PATH.stat().st_size / 1e9:.1f} GB)")
    else:
        k7_lazy = None
        print(f"  K=7: NOT FOUND at {K7_PATH}")

    # K=8: PyArrow only (exceeds u32 row limit)
    if HAS_K8_DATA:
        k8_pf = pq.ParquetFile(str(K8_PATH))
        k8_meta = k8_pf.metadata
        print(f"  K=8: PyArrow ready ({k8_meta.num_rows:,} rows, "
              f"{k8_meta.num_row_groups} row groups, "
              f"{K8_PATH.stat().st_size / 1e9:.1f} GB)")
    else:
        k8_pf = None
        print(f"  K=8: NOT FOUND at {K8_PATH} (optional)")

    # Combine K=1-6 for analyses that need a single DataFrame
    if data_by_k:
        data_35k_small = pl.concat(list(data_by_k.values()))
        print(f"\nCombined K=1-6: {data_35k_small.height:,} rows")
else:
    print("\n35K results directory not found. Set paths above or mount data drive.")

In [ ]:
# ============================================================================
# CELL 11: Cross-Feature-Group Co-occurrence Matrix (35K)
# ============================================================================
# 7 feature groups: InterPro, GO, EC, Keywords, Taxonomy, Length, pLDDT
# Heatmap: how often features from group X co-occur with features from group Y
# Uses K=2-6 data (eagerly loaded); K=7/K=8 sampled via streaming

FEATURE_GROUPS_35K = [
    "interpro", "go_term", "ec_number", "keyword", "taxonomy", "length_bin", "plddt"
]

GROUP_LABELS = {
    "interpro": "InterPro (12.7K)",
    "go_term": "GO (5.8K)",
    "ec_number": "EC (1.1K)",
    "keyword": "Keywords (15.2K)",
    "taxonomy": "Taxonomy (175)",
    "length_bin": "Length (26)",
    "plddt": "pLDDT (6)",
}

if HAS_35K_RESULTS and data_by_k:
    n_groups = len(FEATURE_GROUPS_35K)
    group_idx = {g: i for i, g in enumerate(FEATURE_GROUPS_35K)}
    cross_cooc = np.zeros((n_groups, n_groups), dtype=np.int64)
    total_itemsets = 0

    # Process K=2-6 (eager) -- K=1 is single items, no co-occurrence
    for k_val in range(2, 7):
        if k_val not in data_by_k:
            continue
        df = data_by_k[k_val]
        # Sample large K levels for tractability
        sample = df if df.height <= 200_000 else df.sample(200_000, seed=42)

        for itemset in sample["itemset"].to_list():
            groups_present = set()
            for item_id in itemset:
                g = id_to_group_35k.get(item_id)
                if g in group_idx:
                    groups_present.add(group_idx[g])
            gp_list = list(groups_present)
            for i in range(len(gp_list)):
                cross_cooc[gp_list[i], gp_list[i]] += 1
                for j in range(i + 1, len(gp_list)):
                    cross_cooc[gp_list[i], gp_list[j]] += 1
                    cross_cooc[gp_list[j], gp_list[i]] += 1
        total_itemsets += sample.height
        print(f"  K={k_val}: processed {sample.height:,} itemsets")

    # K=7: streaming sample
    if k7_lazy is not None:
        try:
            k7_sample = k7_lazy.head(200_000).collect()
            for itemset in k7_sample["itemset"].to_list():
                groups_present = set()
                for item_id in itemset:
                    g = id_to_group_35k.get(item_id)
                    if g in group_idx:
                        groups_present.add(group_idx[g])
                gp_list = list(groups_present)
                for i in range(len(gp_list)):
                    cross_cooc[gp_list[i], gp_list[i]] += 1
                    for j in range(i + 1, len(gp_list)):
                        cross_cooc[gp_list[i], gp_list[j]] += 1
                        cross_cooc[gp_list[j], gp_list[i]] += 1
            total_itemsets += k7_sample.height
            print(f"  K=7: processed {k7_sample.height:,} itemsets (streamed sample)")
        except Exception as e:
            print(f"  K=7: streaming failed ({e}), skipping")

    # Normalize to fraction of sampled itemsets
    cross_frac = cross_cooc / total_itemsets
    labels = [GROUP_LABELS[g] for g in FEATURE_GROUPS_35K]

    fig = go.Figure(data=go.Heatmap(
        z=cross_frac,
        x=labels, y=labels,
        colorscale="Plasma",
        text=[[f"{v:.1%}" for v in row] for row in cross_frac],
        texttemplate="%{text}",
        colorbar=dict(title="Fraction of<br>itemsets"),
    ))
    fig.update_layout(
        title=dict(
            text="Cross-Feature-Group Co-occurrence Matrix (35K features, K=2-7)<br>"
                 f"<sub>Based on {total_itemsets:,} sampled itemsets across K=2-7</sub>",
            font=dict(size=16),
        ),
        template=PLOTLY_TEMPLATE, height=650, width=750,
    )
    fig.show()

    print(f"\nTotal sampled: {total_itemsets:,} itemsets")
    print(f"Diagonal = self-occurrence (feature group present in itemset)")
    print(f"Off-diagonal = cross-group co-occurrence")
else:
    print("35K results not loaded. Run Cell 10 first.")

In [ ]:
# ============================================================================
# CELL 12: Feature Group Contribution per K-level (35K)
# ============================================================================
# Stacked bar chart: what % of each K's itemsets contain InterPro vs GO vs EC...
# Shows how pattern composition changes with depth.

# === 1K VERSION (existing Pfam + GO + pLDDT data) ===
GROUPS_1K = ["pfam", "go_term", "plddt_mean", "plddt_fraction"]
id_to_group_1k = {
    row["item_id"]: row["feature_category"]
    for row in mapping.iter_rows(named=True)
}

k_group_data = []
for k_val in range(1, 23):
    k_items = godmode.filter(pl.col("k") == k_val)
    if k_items.height == 0:
        continue
    sample = k_items if k_items.height <= 50_000 else k_items.sample(50_000, seed=42)
    group_counts = {g: 0 for g in GROUPS_1K}
    for itemset in sample["itemset"].to_list():
        groups_seen = set()
        for item_id in itemset:
            g = id_to_group_1k.get(item_id)
            if g:
                groups_seen.add(g)
        for g in groups_seen:
            if g in group_counts:
                group_counts[g] += 1
    n = sample.height
    for g, c in group_counts.items():
        k_group_data.append({"k": k_val, "group": g, "fraction": c / n, "count": c})

kgdf = pl.DataFrame(k_group_data).to_pandas()

fig = go.Figure()
for group in GROUPS_1K:
    subset = kgdf[kgdf["group"] == group]
    fig.add_trace(go.Bar(
        x=subset["k"], y=subset["fraction"] * 100,
        name=group,
        marker_color=CATEGORY_COLORS.get(group, "#94a3b8"),
    ))
fig.update_layout(
    title=dict(
        text="Feature Group Prevalence per K-Level (1K Features)<br>"
             "<sub>% of itemsets at each K that contain at least one feature from each group</sub>",
        font=dict(size=16),
    ),
    barmode="group",
    xaxis=dict(title="K (itemset size)", dtick=1),
    yaxis=dict(title="% of itemsets containing group", range=[0, 105]),
    template=PLOTLY_TEMPLATE, height=500, width=1100,
)
fig.show()

# === 35K VERSION (7 groups) ===
if HAS_35K_RESULTS and data_by_k:
    k_group_data_35k = []

    # K=1-6: eagerly loaded
    for k_val in range(1, 7):
        if k_val not in data_by_k:
            continue
        df = data_by_k[k_val]
        sample = df if df.height <= 50_000 else df.sample(50_000, seed=42)
        group_counts = {g: 0 for g in FEATURE_GROUPS_35K}
        for itemset in sample["itemset"].to_list():
            groups_seen = set()
            for item_id in itemset:
                g = id_to_group_35k.get(item_id)
                if g:
                    groups_seen.add(g)
            for g in groups_seen:
                if g in group_counts:
                    group_counts[g] += 1
        n = sample.height
        for g, c in group_counts.items():
            k_group_data_35k.append({"k": k_val, "group": g, "fraction": c / n})

    # K=7: streaming sample
    if k7_lazy is not None:
        try:
            k7_sample = k7_lazy.head(50_000).collect()
            group_counts = {g: 0 for g in FEATURE_GROUPS_35K}
            for itemset in k7_sample["itemset"].to_list():
                groups_seen = set()
                for item_id in itemset:
                    g = id_to_group_35k.get(item_id)
                    if g:
                        groups_seen.add(g)
                for g in groups_seen:
                    if g in group_counts:
                        group_counts[g] += 1
            n = k7_sample.height
            for g, c in group_counts.items():
                k_group_data_35k.append({"k": 7, "group": g, "fraction": c / n})
        except Exception as e:
            print(f"  K=7 streaming failed: {e}")

    # K=8: PyArrow row-group sample (first row group only)
    if k8_pf is not None:
        try:
            rg0 = k8_pf.read_row_group(0).to_pandas()
            sample_size = min(50_000, len(rg0))
            rg0_sample = rg0.head(sample_size)
            group_counts = {g: 0 for g in FEATURE_GROUPS_35K}
            for itemset in rg0_sample["itemset"]:
                groups_seen = set()
                for item_id in itemset:
                    g = id_to_group_35k.get(item_id)
                    if g:
                        groups_seen.add(g)
                for g in groups_seen:
                    if g in group_counts:
                        group_counts[g] += 1
            for g, c in group_counts.items():
                k_group_data_35k.append({"k": 8, "group": g, "fraction": c / sample_size})
            del rg0, rg0_sample
        except Exception as e:
            print(f"  K=8 PyArrow failed: {e}")

    kgdf35 = pl.DataFrame(k_group_data_35k).to_pandas()
    fig35 = go.Figure()
    for group in FEATURE_GROUPS_35K:
        subset = kgdf35[kgdf35["group"] == group]
        fig35.add_trace(go.Bar(
            x=subset["k"], y=subset["fraction"] * 100,
            name=GROUP_LABELS.get(group, group),
            marker_color=CATEGORY_COLORS.get(group, "#94a3b8"),
        ))
    fig35.update_layout(
        title=dict(
            text="Feature Group Prevalence per K-Level (35K Features, K=1-8)<br>"
                 "<sub>% of itemsets at each K that contain at least one feature from each group</sub>",
            font=dict(size=16),
        ),
        barmode="group",
        xaxis=dict(title="K", dtick=1),
        yaxis=dict(title="% of itemsets", range=[0, 105]),
        template=PLOTLY_TEMPLATE, height=500, width=1200,
    )
    fig35.show()

In [ ]:
# ============================================================================
# CELL 13: Novel Pattern Detector (35K)
# ============================================================================
# Find itemsets containing features from 3+ different feature groups.
# These cross-domain discoveries represent genuinely new science.
# With 7 groups in 35K, patterns spanning 4+ groups are the real prize.

# === 1K VERSION (demonstrates the approach) ===
def count_groups(itemset, lookup):
    """Count distinct feature groups in an itemset."""
    groups = set()
    for item_id in itemset:
        info = lookup.get(item_id)
        if info:
            cat = info[1]
            if cat.startswith("plddt"):
                cat = "plddt"
            groups.add(cat)
    return groups

multi_group_by_k = {}
SAMPLE = 100_000

for k_val in range(2, 23):
    k_items = godmode.filter(pl.col("k") == k_val)
    if k_items.height == 0:
        continue
    sample = k_items if k_items.height <= SAMPLE else k_items.sample(SAMPLE, seed=42)
    n_multi = 0
    n_all_three = 0
    for itemset in sample["itemset"].to_list():
        groups = count_groups(itemset, item_lookup)
        if len(groups) >= 2:
            n_multi += 1
        if len(groups) >= 3:
            n_all_three += 1
    multi_group_by_k[k_val] = {
        "n_total": sample.height,
        "n_multi": n_multi,
        "n_all_three": n_all_three,
        "pct_multi": n_multi / sample.height * 100,
        "pct_all_three": n_all_three / sample.height * 100,
    }

mg_df = pl.DataFrame([{"k": k, **v} for k, v in multi_group_by_k.items()]).to_pandas()

fig = go.Figure()
fig.add_trace(go.Bar(x=mg_df["k"], y=mg_df["pct_multi"],
    name="2+ groups (cross-domain)", marker_color="#a78bfa"))
fig.add_trace(go.Bar(x=mg_df["k"], y=mg_df["pct_all_three"],
    name="3 groups (Pfam+GO+pLDDT)", marker_color="#f472b6"))
fig.update_layout(
    title=dict(text="Cross-Domain Patterns per K-Level (1K Features)<br>"
        "<sub>% of itemsets spanning multiple feature groups (Pfam, GO, pLDDT)</sub>",
        font=dict(size=16)),
    barmode="overlay",
    xaxis=dict(title="K", dtick=1),
    yaxis=dict(title="% of itemsets", range=[0, 105]),
    template=PLOTLY_TEMPLATE, height=450, width=1000,
)
fig.show()

print("\nCross-domain pattern prevalence (1K):")
for _, row in mg_df.iterrows():
    print(f"  K={int(row['k']):>2}: {row['pct_multi']:.1f}% cross-domain, {row['pct_all_three']:.1f}% all 3 groups")

# === 35K VERSION (7 groups -- the real prize) ===
if HAS_35K_RESULTS and data_by_k:
    def count_groups_35k(itemset):
        groups = set()
        for item_id in itemset:
            info = item_lookup_35k.get(item_id)
            if info:
                groups.add(info[1])
        return groups

    multi_group_by_k_35k = {}

    # K=2-6: eager
    for k_val in range(2, 7):
        if k_val not in data_by_k:
            continue
        df = data_by_k[k_val]
        sample = df if df.height <= SAMPLE else df.sample(SAMPLE, seed=42)
        bins = {i: 0 for i in range(1, 8)}
        for itemset in sample["itemset"].to_list():
            ng = len(count_groups_35k(itemset))
            if ng in bins:
                bins[ng] += 1
        multi_group_by_k_35k[k_val] = {"n": sample.height, "bins": bins}
        print(f"  K={k_val}: {sample.height:,} sampled")

    # K=7: streaming
    if k7_lazy is not None:
        try:
            k7_s = k7_lazy.head(SAMPLE).collect()
            bins = {i: 0 for i in range(1, 8)}
            for itemset in k7_s["itemset"].to_list():
                ng = len(count_groups_35k(itemset))
                if ng in bins:
                    bins[ng] += 1
            multi_group_by_k_35k[7] = {"n": k7_s.height, "bins": bins}
            print(f"  K=7: {k7_s.height:,} streamed")
        except Exception as e:
            print(f"  K=7 failed: {e}")

    # K=8: PyArrow sample
    if k8_pf is not None:
        try:
            rg0 = k8_pf.read_row_group(0).to_pandas()
            rg0_s = rg0.head(SAMPLE)
            bins = {i: 0 for i in range(1, 8)}
            for itemset in rg0_s["itemset"]:
                ng = len(count_groups_35k(itemset))
                if ng in bins:
                    bins[ng] += 1
            multi_group_by_k_35k[8] = {"n": len(rg0_s), "bins": bins}
            print(f"  K=8: {len(rg0_s):,} from row group 0")
            del rg0, rg0_s
        except Exception as e:
            print(f"  K=8 failed: {e}")

    # Build visualization
    rows_35k = []
    for k_val, info in sorted(multi_group_by_k_35k.items()):
        n = info["n"]
        for ng, cnt in info["bins"].items():
            rows_35k.append({"k": k_val, "n_groups": ng, "pct": cnt / n * 100})

    mg35_df = pl.DataFrame(rows_35k).to_pandas()

    fig35 = go.Figure()
    group_thresholds = [("2+", 2), ("3+", 3), ("4+", 4), ("5+", 5), ("6+", 6)]
    colors_thresh = ["#a78bfa", "#f472b6", "#fb923c", "#34d399", "#f87171"]

    for (label, thresh), color in zip(group_thresholds, colors_thresh):
        y_vals = []
        k_vals = sorted(multi_group_by_k_35k.keys())
        for k_val in k_vals:
            info = multi_group_by_k_35k[k_val]
            n = info["n"]
            cnt = sum(info["bins"].get(i, 0) for i in range(thresh, 8))
            y_vals.append(cnt / n * 100)
        fig35.add_trace(go.Bar(x=k_vals, y=y_vals, name=f"{label} groups", marker_color=color))

    fig35.update_layout(
        title=dict(
            text="Cross-Domain Patterns per K-Level (35K Features, K=2-8)<br>"
                 "<sub>% of itemsets spanning N+ different feature groups (out of 7 possible)</sub>",
            font=dict(size=16),
        ),
        barmode="group",
        xaxis=dict(title="K", dtick=1),
        yaxis=dict(title="% of itemsets", range=[0, 105]),
        template=PLOTLY_TEMPLATE, height=500, width=1100,
    )
    fig35.show()

    # Find the most diverse patterns (5+ groups)
    print("\nMost diverse cross-domain patterns (5+ feature groups):")
    novel_patterns = []
    # Search in K=5-6 (eager, manageable)
    for k_val in range(5, 7):
        if k_val not in data_by_k:
            continue
        df = data_by_k[k_val]
        sample = df if df.height <= 50_000 else df.sample(50_000, seed=42)
        for row in sample.iter_rows(named=True):
            groups = count_groups_35k(row["itemset"])
            if len(groups) >= 5:
                decoded = [item_lookup_35k.get(x, (f"item_{x}", "?"))[0] for x in row["itemset"]]
                novel_patterns.append({
                    "k": len(row["itemset"]),
                    "n_groups": len(groups),
                    "groups": sorted(groups),
                    "support": row["support"],
                    "features": decoded,
                })

    novel_patterns.sort(key=lambda x: (-x["n_groups"], -x["support"]))
    for p in novel_patterns[:20]:
        print(f"  K={p['k']}, groups={p['n_groups']}: {', '.join(p['groups'])}")
        print(f"    support={p['support']:.2e}, features: {', '.join(p['features'][:8])}...")
    if not novel_patterns:
        print("  (none found at K=5-6 -- check K=7/K=8 for deeper cross-domain patterns)")

In [ ]:
# ============================================================================
# CELL 14: Taxonomy Enrichment (35K)
# ============================================================================
# Are certain taxonomic lineages enriched in high-K patterns?
# Uses the 175 taxonomy features from the 35K feature set.

if ITEM_MAPPING_35K.exists():
    tax_features = mapping_35k.filter(pl.col("feature_category") == "taxonomy")
    print(f"Taxonomy features available: {tax_features.height}")
    print("\nSample taxonomy features:")
    print(tax_features.head(20).select("feature_name", "count"))

if HAS_35K_RESULTS and data_by_k:
    tax_ids = set(
        mapping_35k.filter(pl.col("feature_category") == "taxonomy")["item_id"].to_list()
    )
    tax_name = {
        row["item_id"]: row["feature_name"]
        for row in mapping_35k.filter(
            pl.col("feature_category") == "taxonomy"
        ).iter_rows(named=True)
    }

    # Per-K taxonomy prevalence
    tax_by_k: dict[int, dict[str, float]] = {}

    # K=1-6: eager
    for k_val in range(1, 7):
        if k_val not in data_by_k:
            continue
        df = data_by_k[k_val]
        sample = df if df.height <= 50_000 else df.sample(50_000, seed=42)
        tax_counts: dict[str, int] = {}
        for itemset in sample["itemset"].to_list():
            for item_id in itemset:
                if item_id in tax_ids:
                    name = tax_name[item_id]
                    tax_counts[name] = tax_counts.get(name, 0) + 1
        tax_by_k[k_val] = {name: cnt / sample.height for name, cnt in tax_counts.items()}

    # K=7: streaming sample
    if k7_lazy is not None:
        try:
            k7_sample = k7_lazy.head(50_000).collect()
            tax_counts = {}
            for itemset in k7_sample["itemset"].to_list():
                for item_id in itemset:
                    if item_id in tax_ids:
                        name = tax_name[item_id]
                        tax_counts[name] = tax_counts.get(name, 0) + 1
            tax_by_k[7] = {name: cnt / k7_sample.height for name, cnt in tax_counts.items()}
        except Exception as e:
            print(f"  K=7 taxonomy failed: {e}")

    # K=8: PyArrow sample
    if k8_pf is not None:
        try:
            rg0 = k8_pf.read_row_group(0).to_pandas()
            sample_size = min(50_000, len(rg0))
            rg0_s = rg0.head(sample_size)
            tax_counts = {}
            for itemset in rg0_s["itemset"]:
                for item_id in itemset:
                    if item_id in tax_ids:
                        name = tax_name[item_id]
                        tax_counts[name] = tax_counts.get(name, 0) + 1
            tax_by_k[8] = {name: cnt / sample_size for name, cnt in tax_counts.items()}
            del rg0, rg0_s
        except Exception as e:
            print(f"  K=8 taxonomy failed: {e}")

    # Find top taxonomic lineages
    all_taxa: set[str] = set()
    for v in tax_by_k.values():
        all_taxa.update(v.keys())

    top_taxa = sorted(
        all_taxa,
        key=lambda t: max(tax_by_k.get(k, {}).get(t, 0) for k in tax_by_k),
        reverse=True,
    )[:15]

    fig = go.Figure()
    for taxon in top_taxa:
        k_vals = sorted(tax_by_k.keys())
        fracs = [tax_by_k[k].get(taxon, 0) * 100 for k in k_vals]
        fig.add_trace(go.Scatter(
            x=k_vals, y=fracs, mode="lines+markers", name=taxon,
        ))
    fig.update_layout(
        title=dict(
            text="Taxonomy Enrichment per K-Level (35K Features)<br>"
                 "<sub>Which lineages dominate high-K patterns? Top 15 taxa by max prevalence</sub>",
            font=dict(size=16),
        ),
        xaxis=dict(title="K", dtick=1),
        yaxis=dict(title="% of itemsets containing taxon"),
        template=PLOTLY_TEMPLATE, height=600, width=1100,
        legend=dict(font=dict(size=9)),
    )
    fig.show()

    # Print enrichment summary
    print("\nTaxonomy enrichment summary (max prevalence across K):")
    for taxon in top_taxa:
        max_k = max(tax_by_k.keys(), key=lambda k: tax_by_k[k].get(taxon, 0))
        max_pct = tax_by_k[max_k].get(taxon, 0) * 100
        print(f"  {taxon:40s}: {max_pct:>5.1f}% at K={max_k}")
else:
    print("Taxonomy enrichment requires 35K mining results. Run Cell 10 first.")

In [ ]:
# ============================================================================
# CELL 15: pLDDT Quality Filter (35K)
# ============================================================================
# Compare K-distributions for high-confidence structures (pLDDT > 90) vs all.

# === 1K VERSION ===
PLDDT_HIGH_ID = 2  # plddt_mean_high

high_conf = godmode.filter(pl.col("itemset").list.contains(PLDDT_HIGH_ID))
low_conf = godmode.filter(~pl.col("itemset").list.contains(PLDDT_HIGH_ID))

print(f"High-confidence (pLDDT high): {high_conf.height:,} itemsets ({high_conf.height / godmode.height * 100:.1f}%)")
print(f"Other:                        {low_conf.height:,} itemsets ({low_conf.height / godmode.height * 100:.1f}%)")

k_high = high_conf.group_by("k").agg(pl.len().alias("count_high")).sort("k")
k_low = low_conf.group_by("k").agg(pl.len().alias("count_other")).sort("k")
k_all = godmode.group_by("k").agg(pl.len().alias("count_all")).sort("k")

merged = k_all.join(k_high, on="k", how="left").join(k_low, on="k", how="left").fill_null(0)
merged_pd = merged.to_pandas()

fig = go.Figure()
fig.add_trace(go.Bar(x=merged_pd["k"], y=merged_pd["count_all"],
    name="All itemsets", marker_color="#60a5fa", opacity=0.5))
fig.add_trace(go.Bar(x=merged_pd["k"], y=merged_pd["count_high"],
    name="High pLDDT only", marker_color="#34d399", opacity=0.8))
fig.update_layout(
    title=dict(
        text="K-Distribution: High pLDDT Structures vs All (1K Features)<br>"
             "<sub>Itemsets containing plddt_mean_high feature</sub>",
        font=dict(size=16),
    ),
    barmode="overlay",
    xaxis=dict(title="K", dtick=1),
    yaxis=dict(title="Itemset count", type="log"),
    template=PLOTLY_TEMPLATE, height=500, width=1100,
)
fig.show()

# === 35K VERSION ===
if HAS_35K_RESULTS and data_by_k and ITEM_MAPPING_35K.exists():
    # Find pLDDT high item IDs in 35K mapping
    plddt_feats = mapping_35k.filter(pl.col("feature_category") == "plddt")
    print("\n35K pLDDT features:")
    print(plddt_feats.select("item_id", "feature_name", "count"))

    # Identify "high" pLDDT features (mean_high or frac_high)
    plddt_high_ids_35k = set(
        plddt_feats.filter(
            pl.col("feature_name").str.contains("high")
        )["item_id"].to_list()
    )
    print(f"\nHigh pLDDT item IDs: {plddt_high_ids_35k}")

    if plddt_high_ids_35k:
        k_stats_35k = []

        for k_val in range(1, 7):
            if k_val not in data_by_k:
                continue
            df = data_by_k[k_val]
            n_total = df.height

            # Filter for itemsets containing any high pLDDT feature
            n_high = 0
            for pid in plddt_high_ids_35k:
                n_high += df.filter(pl.col("itemset").list.contains(pid)).height

            # Deduplicate: use set-based counting on a sample
            sample = df if df.height <= 100_000 else df.sample(100_000, seed=42)
            n_with_high = 0
            for itemset in sample["itemset"].to_list():
                if plddt_high_ids_35k.intersection(itemset):
                    n_with_high += 1
            frac_high = n_with_high / sample.height

            k_stats_35k.append({
                "k": k_val,
                "n_total": n_total,
                "frac_high_plddt": frac_high,
                "n_high_est": int(n_total * frac_high),
            })

        # K=7: streaming
        if k7_lazy is not None:
            try:
                k7_s = k7_lazy.head(100_000).collect()
                n_with_high = sum(
                    1 for itemset in k7_s["itemset"].to_list()
                    if plddt_high_ids_35k.intersection(itemset)
                )
                k_stats_35k.append({
                    "k": 7,
                    "n_total": 3_507_040_364,  # known count
                    "frac_high_plddt": n_with_high / k7_s.height,
                    "n_high_est": int(3_507_040_364 * n_with_high / k7_s.height),
                })
            except Exception as e:
                print(f"  K=7 pLDDT filter failed: {e}")

        ks_df = pl.DataFrame(k_stats_35k).to_pandas()

        fig35 = make_subplots(specs=[[{"secondary_y": True}]])
        fig35.add_trace(go.Bar(
            x=ks_df["k"], y=ks_df["n_total"],
            name="All itemsets", marker_color="#60a5fa", opacity=0.4,
        ), secondary_y=False)
        fig35.add_trace(go.Bar(
            x=ks_df["k"], y=ks_df["n_high_est"],
            name="High pLDDT subset", marker_color="#34d399", opacity=0.8,
        ), secondary_y=False)
        fig35.add_trace(go.Scatter(
            x=ks_df["k"], y=ks_df["frac_high_plddt"] * 100,
            name="% high pLDDT", mode="lines+markers",
            line=dict(color="#f472b6", width=3),
        ), secondary_y=True)

        fig35.update_layout(
            title=dict(
                text="pLDDT Quality Filter: High-Confidence Patterns (35K Features)<br>"
                     "<sub>What fraction of patterns at each K involve high-confidence structures?</sub>",
                font=dict(size=16),
            ),
            barmode="overlay",
            template=PLOTLY_TEMPLATE, height=550, width=1100,
        )
        fig35.update_xaxes(title_text="K", dtick=1)
        fig35.update_yaxes(title_text="Itemset count", type="log", secondary_y=False)
        fig35.update_yaxes(title_text="% with high pLDDT", secondary_y=True)
        fig35.show()

        print("\npLDDT quality summary:")
        for _, row in ks_df.iterrows():
            print(f"  K={int(row['k'])}: {row['frac_high_plddt']*100:.1f}% high pLDDT "
                  f"({int(row['n_high_est']):,} / {int(row['n_total']):,})")

---

# Part 2.5: K=8 Summary Statistics

Aggregate view across all K levels (K=1 through K=8). Total: **16,812,646,639** frequent itemsets.

---

In [ ]:
# ============================================================================
# CELL 15.5: K=8 Summary Statistics
# ============================================================================
# Aggregate view: count by K, support distributions, growth rates.
# K=1-6 from eager DataFrames, K=7 from streaming, K=8 from PyArrow metadata.

# --- Known counts (from mining logs) ---
KNOWN_COUNTS = {
    1: 28_405,
    2: 5_506_372,
    3: 176_048_236,
    4: 1_506_508_703,
    5: 2_474_423_427,
    6: 626_781_137,
    7: 3_507_040_364,
    8: 12_072_309_005,
}
TOTAL_ITEMSETS = sum(KNOWN_COUNTS.values())  # 16,812,646,639

# ---- Panel 1: Count by K level ----
k_levels = sorted(KNOWN_COUNTS.keys())
counts = [KNOWN_COUNTS[k] for k in k_levels]

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Itemset Count by K Level",
        "Growth Rate: K[n] / K[n-1]",
        "Support Distribution per K (box plot)",
        "Cumulative Itemsets",
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.10,
)

# Panel 1: Bar chart of counts per K
fig.add_trace(go.Bar(
    x=k_levels, y=counts,
    marker_color=["#60a5fa"] * 6 + ["#fb923c", "#f87171"],
    text=[f"{c/1e9:.2f}B" if c > 1e9 else f"{c/1e6:.0f}M" if c > 1e6 else f"{c:,}"
          for c in counts],
    textposition="outside",
    showlegend=False,
), row=1, col=1)
fig.update_yaxes(type="log", title_text="Count (log)", row=1, col=1)
fig.update_xaxes(title_text="K", dtick=1, row=1, col=1)

# Panel 2: Growth rate K[n] / K[n-1]
growth_rates = []
for i in range(1, len(k_levels)):
    ratio = KNOWN_COUNTS[k_levels[i]] / KNOWN_COUNTS[k_levels[i - 1]]
    growth_rates.append(ratio)

fig.add_trace(go.Scatter(
    x=k_levels[1:], y=growth_rates,
    mode="lines+markers+text",
    marker=dict(size=10, color="#a78bfa"),
    line=dict(width=3, color="#a78bfa"),
    text=[f"{r:.1f}x" for r in growth_rates],
    textposition="top center",
    showlegend=False,
), row=1, col=2)
fig.add_hline(y=1.0, line_dash="dash", line_color="#475569", row=1, col=2)
fig.update_yaxes(title_text="Growth ratio", type="log", row=1, col=2)
fig.update_xaxes(title_text="K", dtick=1, row=1, col=2)

# Panel 3: Support distribution per K (box plot from sampled data)
if HAS_35K_RESULTS and data_by_k:
    for k_val in range(1, 7):
        if k_val not in data_by_k:
            continue
        df = data_by_k[k_val]
        sample = df if df.height <= 10_000 else df.sample(10_000, seed=42)
        supports = sample["support"].to_list()
        fig.add_trace(go.Box(
            y=supports, name=f"K={k_val}",
            marker_color="#60a5fa" if k_val <= 6 else "#fb923c",
            showlegend=False,
        ), row=2, col=1)

    # K=7: streaming sample
    if k7_lazy is not None:
        try:
            k7_box = k7_lazy.select("support").head(10_000).collect()
            fig.add_trace(go.Box(
                y=k7_box["support"].to_list(), name="K=7",
                marker_color="#fb923c", showlegend=False,
            ), row=2, col=1)
        except Exception:
            pass

    # K=8: PyArrow sample for support distribution
    if k8_pf is not None:
        try:
            rg0 = k8_pf.read_row_group(0)
            supports_k8 = rg0.column("support").to_pylist()[:10_000]
            fig.add_trace(go.Box(
                y=supports_k8, name="K=8",
                marker_color="#f87171", showlegend=False,
            ), row=2, col=1)
            del rg0
        except Exception:
            pass

fig.update_yaxes(title_text="Support", type="log", row=2, col=1)

# Panel 4: Cumulative itemsets
cumulative = []
running = 0
for k in k_levels:
    running += KNOWN_COUNTS[k]
    cumulative.append(running)

fig.add_trace(go.Scatter(
    x=k_levels, y=cumulative,
    mode="lines+markers",
    fill="tozeroy",
    marker=dict(size=8, color="#34d399"),
    line=dict(width=3, color="#34d399"),
    showlegend=False,
), row=2, col=2)
fig.add_trace(go.Scatter(
    x=k_levels, y=[TOTAL_ITEMSETS] * len(k_levels),
    mode="lines", line=dict(dash="dash", color="#f87171"),
    name=f"Total: {TOTAL_ITEMSETS/1e9:.2f}B",
    showlegend=True,
), row=2, col=2)
fig.update_yaxes(title_text="Cumulative count", row=2, col=2)
fig.update_xaxes(title_text="K", dtick=1, row=2, col=2)

fig.update_layout(
    title=dict(
        text="K=8 Mining Summary: 16.8 Billion Frequent Itemsets<br>"
             "<sub>35K features, 109.2M proteins, K=1-8 complete</sub>",
        font=dict(size=18),
    ),
    template=PLOTLY_TEMPLATE,
    height=900, width=1200,
)
fig.show()

# --- Print summary table ---
print("\n" + "=" * 70)
print("K=8 MINING SUMMARY")
print("=" * 70)
print(f"{'K':<5} {'Count':>18} {'% of Total':>12} {'Growth':>10}")
print("-" * 50)
for k in k_levels:
    pct = KNOWN_COUNTS[k] / TOTAL_ITEMSETS * 100
    growth = f"{KNOWN_COUNTS[k] / KNOWN_COUNTS[k-1]:.1f}x" if k > 1 else "--"
    print(f"{k:<5} {KNOWN_COUNTS[k]:>18,} {pct:>11.2f}% {growth:>10}")
print("-" * 50)
print(f"{'TOTAL':<5} {TOTAL_ITEMSETS:>18,} {100.0:>11.2f}%")
print()
print(f"K=8 alone: {KNOWN_COUNTS[8]/TOTAL_ITEMSETS*100:.1f}% of all itemsets")
print(f"K=7+K=8:   {(KNOWN_COUNTS[7]+KNOWN_COUNTS[8])/TOTAL_ITEMSETS*100:.1f}% of all itemsets")
print(f"The K=8 explosion ({KNOWN_COUNTS[8]/KNOWN_COUNTS[7]:.1f}x from K=7) suggests K=9 could be 10x+ larger.")

---

# Part 3: Association Rule Mining & Network Analysis

Mining frequent itemsets tells us **what co-occurs**. Association rules tell us **what implies what** -- with directionality, confidence, and lift.

---

In [ ]:
# ============================================================================
# CELL 16: Generate Association Rules (1K + 35K)
# ============================================================================
# 1K: existing approach on godmode data
# 35K: Top-N approach -- sort K=4 by support, take top 10K, generate rules

import sys
sys.path.insert(0, str(REPO_ROOT / "src"))

from et_miner.rules import generate_rules, Rule

# === 1K RULES (existing) ===
rule_candidates = godmode.filter(
    (pl.col("k") >= 2) & (pl.col("k") <= 6) & (pl.col("support") > 1e-5)
).select("itemset", "support")

print(f"[1K] Rule candidates: {rule_candidates.height:,} itemsets (K=2-6, support > 1e-5)")
print("Generating rules (min_confidence=0.5)...")

rules = generate_rules(rule_candidates, min_confidence=0.5)
print(f"[1K] Generated {len(rules):,} rules")

# Convert to DataFrame for analysis
rules_data = []
for r in rules:
    lhs_names = [item_lookup.get(x, (f"item_{x}", "?"))[0] for x in r.lhs]
    rhs_names = [item_lookup.get(x, (f"item_{x}", "?"))[0] for x in r.rhs]
    lhs_cats = [item_lookup.get(x, (f"item_{x}", "?"))[1] for x in r.lhs]
    rhs_cats = [item_lookup.get(x, (f"item_{x}", "?"))[1] for x in r.rhs]

    rules_data.append({
        "lhs": " + ".join(lhs_names),
        "rhs": " + ".join(rhs_names),
        "lhs_categories": set(lhs_cats),
        "rhs_categories": set(rhs_cats),
        "support": r.support,
        "confidence": r.confidence,
        "lift": r.lift,
        "lhs_ids": r.lhs,
        "rhs_ids": r.rhs,
        "is_cross_group": set(lhs_cats) != set(rhs_cats),
        "source": "1k",
    })

print(f"\n[1K] Rule statistics:")
print(f"  Total rules:       {len(rules_data):,}")
print(f"  Cross-group rules: {sum(1 for r in rules_data if r['is_cross_group']):,}")
if rules_data:
    print(f"  Max lift:          {max(r['lift'] for r in rules_data):.1f}")
    print(f"  Mean confidence:   {np.mean([r['confidence'] for r in rules_data]):.3f}")

# === 35K RULES (Top-N approach on K=4) ===
rules_data_35k = []

if HAS_35K_RESULTS and 4 in data_by_k:
    # Sort K=4 by support descending, take top 10K
    k4_top = data_by_k[4].sort("support", descending=True).head(10_000)
    print(f"\n[35K] Top-N rule candidates: {k4_top.height:,} K=4 itemsets (sorted by support)")
    print(f"  Support range: {k4_top['support'].min():.2e} -- {k4_top['support'].max():.2e}")

    # Also need K=1-3 support for the sub-itemsets
    # Build a combined support lookup from K=1-4
    support_frames = []
    for k_val in range(1, 5):
        if k_val in data_by_k:
            support_frames.append(data_by_k[k_val].select("itemset", "support"))
    all_supports = pl.concat(support_frames)

    print(f"  Support lookup: {all_supports.height:,} itemsets")
    print("  Generating 35K rules (min_confidence=0.3)...")

    rules_35k = generate_rules(
        pl.concat([all_supports, k4_top.select("itemset", "support")]).unique("itemset"),
        min_confidence=0.3,
    )
    print(f"[35K] Generated {len(rules_35k):,} rules")

    for r in rules_35k:
        lhs_names = [item_lookup_35k.get(x, (f"item_{x}", "?"))[0] for x in r.lhs]
        rhs_names = [item_lookup_35k.get(x, (f"item_{x}", "?"))[0] for x in r.rhs]
        lhs_cats = [item_lookup_35k.get(x, (f"item_{x}", "?"))[1] for x in r.lhs]
        rhs_cats = [item_lookup_35k.get(x, (f"item_{x}", "?"))[1] for x in r.rhs]

        rules_data_35k.append({
            "lhs": " + ".join(lhs_names),
            "rhs": " + ".join(rhs_names),
            "lhs_categories": set(lhs_cats),
            "rhs_categories": set(rhs_cats),
            "support": r.support,
            "confidence": r.confidence,
            "lift": r.lift,
            "lhs_ids": r.lhs,
            "rhs_ids": r.rhs,
            "is_cross_group": set(lhs_cats) != set(rhs_cats),
            "source": "35k",
        })

    print(f"\n[35K] Rule statistics:")
    print(f"  Total rules:       {len(rules_data_35k):,}")
    print(f"  Cross-group rules: {sum(1 for r in rules_data_35k if r['is_cross_group']):,}")
    if rules_data_35k:
        print(f"  Max lift:          {max(r['lift'] for r in rules_data_35k):.1f}")
        print(f"  Mean confidence:   {np.mean([r['confidence'] for r in rules_data_35k]):.3f}")

# Show top 20 by lift (1K)
print("\n" + "=" * 110)
print("Top 20 Rules by Lift (1K):")
print("-" * 110)
sorted_rules = sorted(rules_data, key=lambda r: -r["lift"])
for i, r in enumerate(sorted_rules[:20]):
    cross = "*" if r["is_cross_group"] else " "
    print(f"{cross} {r['lhs']:<45} => {r['rhs']:<30} lift={r['lift']:>8.1f}  conf={r['confidence']:.3f}  sup={r['support']:.2e}")

if rules_data_35k:
    print("\nTop 20 Rules by Lift (35K):")
    print("-" * 110)
    sorted_35k = sorted(rules_data_35k, key=lambda r: -r["lift"])
    for i, r in enumerate(sorted_35k[:20]):
        cross = "*" if r["is_cross_group"] else " "
        print(f"{cross} {r['lhs']:<45} => {r['rhs']:<30} lift={r['lift']:>8.1f}  conf={r['confidence']:.3f}  sup={r['support']:.2e}")

In [ ]:
# ============================================================================
# CELL 17: Top Rules by Lift -- Visualization (1K + 35K)
# ============================================================================
# Focus on cross-feature-group rules

def plot_top_rules(rules_list, title_suffix="", top_n=50):
    """Plot top cross-group rules as confidence vs lift scatter."""
    cross_rules = [r for r in rules_list if r["is_cross_group"]]
    cross_rules.sort(key=lambda r: -r["lift"])
    top_n = min(top_n, len(cross_rules))
    top_cross = cross_rules[:top_n]

    if not top_cross:
        print(f"No cross-group rules found {title_suffix}.")
        return

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=[r["confidence"] for r in top_cross],
        y=[r["lift"] for r in top_cross],
        mode="markers",
        marker=dict(
            size=[max(5, min(30, np.log10(1 / r["support"]) * 3)) for r in top_cross],
            color=[r["lift"] for r in top_cross],
            colorscale="Plasma",
            colorbar=dict(title="Lift"),
            opacity=0.7,
        ),
        text=[f"{r['lhs']} => {r['rhs']}" for r in top_cross],
        hovertemplate=(
            "%{text}<br>"
            "Confidence: %{x:.3f}<br>"
            "Lift: %{y:.1f}<br>"
            "Support: %{customdata:.2e}<extra></extra>"
        ),
        customdata=[r["support"] for r in top_cross],
    ))
    fig.add_hline(y=1.0, line_dash="dash", line_color="#475569",
                  annotation_text="Lift=1 (independence)")
    fig.update_layout(
        title=dict(
            text=f"Top {top_n} Cross-Group Association Rules {title_suffix}<br>"
                 f"<sub>Bubble size = -log(support)</sub>",
            font=dict(size=16),
        ),
        xaxis=dict(title="Confidence (P(RHS|LHS))"),
        yaxis=dict(title="Lift (observed/expected co-occurrence)"),
        template=PLOTLY_TEMPLATE, height=600, width=1000,
    )
    fig.show()

    print(f"\nTop {min(15, len(cross_rules))} cross-group rules by lift {title_suffix}:")
    print(f"{'LHS':<40} {'RHS':<25} {'Lift':>8} {'Conf':>6} {'Sup':>10}")
    print("-" * 95)
    for r in cross_rules[:15]:
        print(f"{r['lhs'][:40]:<40} {r['rhs'][:25]:<25} {r['lift']:>8.1f} {r['confidence']:>6.3f} {r['support']:>10.2e}")

# === 1K Rules ===
plot_top_rules(rules_data, title_suffix="(1K Features)")

# === 35K Rules ===
if rules_data_35k:
    plot_top_rules(rules_data_35k, title_suffix="(35K Features)")

In [ ]:
# ============================================================================
# CELL 18: Rule Network Graph (1K + 35K)
# ============================================================================
# Network visualization: nodes = features, edges = rules weighted by lift.
# Color nodes by feature group.

try:
    import networkx as nx
    HAS_NX = True
except ImportError:
    HAS_NX = False
    print("networkx not installed. Install with: uv pip install networkx")

def build_rule_network(rules_list, lookup, title_suffix="", max_rules=200):
    """Build and display a rule network graph."""
    if not HAS_NX or not rules_list:
        return None

    G = nx.DiGraph()
    top_for_graph = sorted(rules_list, key=lambda r: -r["lift"])[:max_rules]
    node_categories = {}

    for r in top_for_graph:
        for lid in r["lhs_ids"]:
            lname, lcat = lookup.get(lid, (f"item_{lid}", "unknown"))
            node_categories[lname] = lcat
            for rid in r["rhs_ids"]:
                rname, rcat = lookup.get(rid, (f"item_{rid}", "unknown"))
                node_categories[rname] = rcat
                if G.has_edge(lname, rname):
                    G[lname][rname]["weight"] = max(G[lname][rname]["weight"], r["lift"])
                else:
                    G.add_edge(lname, rname, weight=r["lift"], confidence=r["confidence"])

    print(f"Network {title_suffix}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

    pos = nx.spring_layout(G, k=2.0, iterations=100, seed=42)

    edge_x, edge_y = [], []
    for u, v, data in G.edges(data=True):
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=edge_x, y=edge_y,
        mode="lines",
        line=dict(width=0.5, color="rgba(150,150,150,0.2)"),
        hoverinfo="none", showlegend=False,
    ))

    for cat in set(node_categories.values()):
        nodes_in_cat = [n for n, c in node_categories.items() if c == cat and n in pos]
        if not nodes_in_cat:
            continue
        fig.add_trace(go.Scatter(
            x=[pos[n][0] for n in nodes_in_cat],
            y=[pos[n][1] for n in nodes_in_cat],
            mode="markers+text",
            marker=dict(
                size=[min(20, 5 + G.degree(n)) for n in nodes_in_cat],
                color=CATEGORY_COLORS.get(cat, "#94a3b8"),
                line=dict(width=1, color="white"),
            ),
            text=nodes_in_cat,
            textposition="top center",
            textfont=dict(size=7, color="#e2e8f0"),
            name=cat,
            hovertemplate="%{text}<br>Degree: %{customdata}<extra></extra>",
            customdata=[G.degree(n) for n in nodes_in_cat],
        ))

    fig.update_layout(
        title=dict(
            text=f"Association Rule Network {title_suffix}<br>"
                 f"<sub>{G.number_of_nodes()} features, {G.number_of_edges()} rules, top {max_rules} by lift</sub>",
            font=dict(size=16),
        ),
        template=PLOTLY_TEMPLATE,
        height=800, width=1000,
        showlegend=True,
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    )
    fig.show()

    hubs = sorted(G.nodes(), key=lambda n: G.degree(n), reverse=True)[:10]
    print(f"\nTop 10 Hub Features {title_suffix}:")
    for h in hubs:
        cat = node_categories.get(h, "?")
        print(f"  {h} ({cat}): degree={G.degree(h)}")

    return G

# === 1K Network ===
if rules_data:
    G_1k = build_rule_network(rules_data, item_lookup, title_suffix="(1K Features)")
else:
    print("No 1K rules generated. Run Cell 16 first.")

# === 35K Network ===
if rules_data_35k:
    G_35k = build_rule_network(rules_data_35k, item_lookup_35k, title_suffix="(35K Features)")
elif HAS_35K_RESULTS:
    print("No 35K rules generated. Run Cell 16 first.")

---

# Summary & Outlook

## Key Findings from the 1K-Feature God Mode Campaign

1. **Scale:** 26.8M itemsets mined in 7.3 minutes across K=1--22, from 76.9M proteins. This is the first proteome-scale frequent itemset mining result ever reported.

2. **Biological hierarchy:** The K-distribution peaks at K=9 (3.53M itemsets), reflecting the typical complexity of protein functional modules. The long tail to K=22 reveals ultra-rare but biologically coherent signatures.

3. **Statistical validation:** The null model (random permutation) produces exactly zero itemsets at K >= 7. All 89,566 patterns at K >= 7 are **impossible** under the null hypothesis -- representing genuine biological co-occurrence.

4. **Method superiority:** Direct GPU mining finds 20.8x more itemsets than the SON approximation algorithm, in 21.4x less time. The CSR bitvector approach makes approximate methods obsolete.

5. **Deep patterns:** The K=19 sentinel (RNA helicase/spliceosome signature) represents a 19-feature functional module found in ~187 proteins. The K=22 apex is a single 22-feature combination found in ~8 proteins -- the deepest co-occurrence ever discovered computationally.

## 35K-Feature Alpha Centauri Campaign: 16.8 Billion Itemsets

6. **Combinatorial explosion:** 35K features produced 16.8B itemsets at K=1--8, compared to 26.8M at 1K features. K=8 alone (12.07B itemsets, 67 GB) represents 71.8% of all itemsets -- the combinatorial frontier is still expanding.

7. **Growth rate dynamics:** K=8/K=7 = 3.4x growth. K=5/K=4 = 1.6x (plateau). K=7/K=6 = 5.6x (re-explosion). The non-monotonic growth pattern suggests distinct biological regimes at different K levels.

8. **Cross-domain patterns:** With 7 feature groups (InterPro, GO, EC, Keywords, Taxonomy, Length, pLDDT), patterns spanning 4+ groups reveal genuine multi-domain biological discoveries impossible with single-ontology approaches.

9. **Taxonomy enrichment:** Lineage-specific patterns at high K reveal which organismal groups have the most complex protein signatures -- connecting structural motifs to evolutionary history.

## What Comes Next

- **K=9+:** The 3.4x growth from K=7 to K=8 suggests K=9 could yield 30--40B+ itemsets. This requires multi-GPU streaming with careful memory management.
- **Association rule mining at scale:** The drop-1 approach (K rules per K-itemset) enables rule generation on billion-row datasets without combinatorial explosion.
- **Drug target nomination:** Scored motifs from the 35K campaign feed directly into the druggability pipeline (score_druggability.py -> nominate_targets.py).

---

*Prepared for review by Prof. Alexandre Bonvin, Utrecht University*
*ET-miner | GPU-accelerated proteome mining | February 2026*